# Good Pot Bad Crop Evaluation

This notebook provides a streamlined evaluation workflow for the `good_pot_bad_crop` use case.

Evaluation plan:
- load MLflow runs for the experiment and reconstruct the evaluated strategy combinations
- treat a **strategy** as `classifier(with params) | actual scorer + actual assigner(with params)`
- inspect run completeness against the generated manifest
- plot learning curves over the logged MLflow step budget
- compute per-run normalized area under the learning curve (AULC) and summarize it in tables
- compute pairwise win/tie/loss comparisons across matched `dataset x seed x step` cells

The notebook assumes the experiment logs come from the `configs/launch/use_cases/good_pot_bad_crop.json` sweep.


In [ ]:
from __future__ import annotations

import itertools
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient

plt.style.use("ggplot")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "configs").exists() else NOTEBOOK_DIR.parent

USE_CASE_NAME = "good_pot_bad_crop"
EXPERIMENT_NAME = "good_pot_bad_crop"
PRIMARY_METRIC = "acc_pair_micro" # "acc_pair_micro" "test_acc"
ONLY_FINISHED = True
DEDUPLICATE_LATEST_PER_SEED = True

results_root = os.environ.get("DALC_RESULTS_ROOT") or os.environ.get("DALCE_RESULTS_ROOT")
RESULTS_PATH = Path(results_root) / "mlflow" if results_root else REPO_ROOT / "mlflow"
MANIFEST_PATH = REPO_ROOT / "manifests" / f"{USE_CASE_NAME}.jsonl"
USE_CASE_PATH = REPO_ROOT / "configs" / "launch" / "use_cases" / f"{USE_CASE_NAME}.json"

print(f"REPO_ROOT      : {REPO_ROOT}")
print(f"RESULTS_PATH   : {RESULTS_PATH}")
print(f"MANIFEST_PATH  : {MANIFEST_PATH}")
print(f"USE_CASE_PATH  : {USE_CASE_PATH}")
print(f"EXPERIMENT_NAME: {EXPERIMENT_NAME}")


In [ ]:
def configure_tracking(results_path: Path) -> MlflowClient:
    results_path = Path(results_path).expanduser().resolve()
    db_path = results_path if results_path.suffix.lower() == ".db" else results_path / "mlruns_0.db"
    tracking_uri = f"sqlite:///{db_path.as_posix()}"
    mlflow.set_tracking_uri(tracking_uri)
    return MlflowClient(tracking_uri=tracking_uri)


def find_experiment(client: MlflowClient, experiment_name: str):
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        raise ValueError(f"MLflow experiment {experiment_name!r} not found.")
    return exp


def search_runs_all(client: MlflowClient, experiment_id: str):
    return client.search_runs(
        [experiment_id],
        run_view_type=ViewType.ALL,
        max_results=50000,
        order_by=["attributes.start_time DESC"],
    )


def runs_to_frame(runs) -> pd.DataFrame:
    rows = []
    for run in runs:
        row = {
            "run_id": run.info.run_id,
            "status": run.info.status,
            "artifact_uri": run.info.artifact_uri,
            "start_time": pd.to_datetime(run.info.start_time, unit="ms", utc=True),
            "end_time": pd.to_datetime(run.info.end_time, unit="ms", utc=True) if run.info.end_time else pd.NaT,
        }
        for k, v in run.data.params.items():
            row[f"param/{k}"] = v
        for k, v in run.data.metrics.items():
            row[f"metric/{k}"] = v
        for k, v in run.data.tags.items():
            row[f"tag/{k}"] = v
        rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("start_time", ascending=False).reset_index(drop=True)
    return df


def count_manifest_rows(path: Path) -> int | None:
    path = Path(path)
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as fh:
        return sum(1 for _ in fh)


def load_use_case(path: Path) -> dict | None:
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def extract_expected_seed_values(use_case: dict | None) -> list[int] | None:
    if not use_case:
        return None
    for axis in use_case.get("axes", []):
        if axis.get("name") == "seed":
            values = axis.get("values")
            if values is None:
                return None
            return [int(v) for v in values]
    return None


def first_present(row: pd.Series, keys, default=None):
    for key in keys:
        if key in row.index and pd.notna(row[key]):
            return row[key]
    return default


def fmt_num(value, digits=3):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "?"
    try:
        f = float(value)
    except Exception:
        return str(value)
    if f.is_integer():
        return str(int(f))
    return f"{f:.{digits}g}"


def derive_classifier_variant(row: pd.Series) -> str:
    choice = first_present(row, ["param/choice/classifier"], default="unknown")
    if choice == "annot_mix_gating":
        p = first_present(row, ["param/classifier/sample_gate_init_prob"])
        return f"annot_mix_gating[p={fmt_num(p)}]"
    if choice == "annot_mix":
        sdim = first_present(row, ["param/classifier/sample_embed_dim"])
        adim = first_present(row, ["param/classifier/annotator_embed_dim"])
        return f"annot_mix[e={fmt_num(sdim)}/{fmt_num(adim)}]"
    if choice == "crowd_em":
        return "crowd_em"
    if choice == "majority_voting":
        return "majority_voting"
    return str(choice)


def derive_scorer_variant(row: pd.Series) -> str:
    return str(first_present(row, ["param/choice/scorer_scorer.actual"], default="unknown"))


def derive_assigner_variant(row: pd.Series) -> str:
    choice = first_present(row, ["param/choice/assigner_assigner.actual"], default="unknown")
    if choice == "epsilon_greedy_sample":
        eps = first_present(row, ["param/assigner/actual/epsilon_min"])
        return f"epsilon_greedy_sample[eps={fmt_num(eps)}]"
    return str(choice)


def enrich_runs(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    out = df.copy()
    out["dataset"] = out.apply(lambda row: first_present(row, ["param/choice/dataset"], default="unknown"), axis=1)
    out["seed"] = pd.to_numeric(out.get("param/seed"), errors="coerce")
    out["classifier_variant"] = out.apply(derive_classifier_variant, axis=1)
    out["scorer_variant"] = out.apply(derive_scorer_variant, axis=1)
    out["assigner_variant"] = out.apply(derive_assigner_variant, axis=1)
    out["strategy"] = out.apply(
        lambda row: f"{row['classifier_variant']} | {row['scorer_variant']} + {row['assigner_variant']}",
        axis=1,
    )
    return out


def apply_run_filters(
    runs_df: pd.DataFrame,
    *,
    filters: dict[str, list] | None = None,
    strategy_contains: list[str] | None = None,
    only_finished: bool | None = None,
) -> pd.DataFrame:
    df = runs_df.copy()
    if only_finished is True and "status" in df.columns:
        df = df[df["status"] == "FINISHED"]
    if only_finished is False and "status" in df.columns:
        df = df[df["status"] != "FINISHED"]

    filters = filters or {}
    for col, values in filters.items():
        if values is None:
            continue
        values = [v for v in values if v is not None and v != ""]
        if not values:
            continue
        if col not in df.columns:
            raise KeyError(f"Unknown filter column: {col}")
        df = df[df[col].isin(values)]

    if strategy_contains:
        mask = pd.Series(True, index=df.index)
        for token in strategy_contains:
            if token:
                mask &= df["strategy"].str.contains(token, case=False, regex=False, na=False)
        df = df[mask]

    return df.reset_index(drop=True)


def summarize_seed_values(seed_series: pd.Series) -> tuple[int, tuple[int, ...]]:
    seeds = pd.Series(seed_series).dropna()
    if seeds.empty:
        return 0, tuple()
    values = tuple(sorted({int(v) for v in seeds.astype(int).tolist()}))
    return len(values), values


def summarize_run_coverage(
    runs_df: pd.DataFrame,
    *,
    group_cols: list[str] | tuple[str, ...] = ("dataset", "strategy"),
) -> pd.DataFrame:
    if runs_df.empty:
        return pd.DataFrame(columns=[*group_cols, "n_runs", "n_seeds", "seeds", "duplicate_runs"])

    rows = []
    for keys, group in runs_df.groupby(list(group_cols), dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        n_seeds, seeds = summarize_seed_values(group["seed"])
        row = dict(zip(group_cols, keys))
        row["n_runs"] = int(group["run_id"].nunique())
        row["n_seeds"] = int(n_seeds)
        row["seeds"] = seeds
        row["duplicate_runs"] = int(row["n_runs"] - row["n_seeds"])
        rows.append(row)

    return pd.DataFrame(rows).sort_values(list(group_cols)).reset_index(drop=True)


def select_latest_seed_runs(
    runs_df: pd.DataFrame,
    *,
    subset_cols: list[str] | tuple[str, ...] = ("dataset", "strategy", "seed"),
) -> pd.DataFrame:
    if runs_df.empty:
        return runs_df.copy()

    order_cols = [col for col in ["end_time", "start_time"] if col in runs_df.columns]
    df = runs_df.sort_values(order_cols + ["run_id"], ascending=[False] * (len(order_cols) + 1)).copy()
    df = df.drop_duplicates(list(subset_cols), keep="first")
    return df.sort_values("start_time", ascending=False).reset_index(drop=True)


In [ ]:
def load_metric_history_table(
    client: MlflowClient,
    runs_df: pd.DataFrame,
    metric_name: str,
    *,
    only_finished: bool = True,
) -> pd.DataFrame:
    rows = []
    subset = runs_df
    if only_finished:
        subset = subset[subset["status"] == "FINISHED"]

    for _, run in subset.iterrows():
        history = client.get_metric_history(run["run_id"], metric_name)
        for item in history:
            rows.append(
                {
                    "run_id": run["run_id"],
                    "dataset": run["dataset"],
                    "seed": run["seed"],
                    "strategy": run["strategy"],
                    "classifier_variant": run["classifier_variant"],
                    "scorer_variant": run["scorer_variant"],
                    "assigner_variant": run["assigner_variant"],
                    "metric": metric_name,
                    "step": int(item.step),
                    "value": float(item.value),
                }
            )
    hist_df = pd.DataFrame(rows)
    if not hist_df.empty:
        hist_df = hist_df.sort_values(["dataset", "strategy", "seed", "step"]).reset_index(drop=True)
    return hist_df


def normalized_aulc(step_values: pd.DataFrame) -> float:
    if step_values.shape[0] == 0:
        return np.nan
    curve = step_values[["step", "value"]].dropna().sort_values("step")
    if curve.empty:
        return np.nan
    if curve.shape[0] == 1:
        return float(curve["value"].iloc[0])
    span = float(curve["step"].iloc[-1] - curve["step"].iloc[0])
    area = float(np.trapezoid(curve["value"].to_numpy(), curve["step"].to_numpy()))
    return area / span if span > 0 else float(curve["value"].iloc[-1])


def compute_aulc_per_run(history_df: pd.DataFrame) -> pd.DataFrame:
    if history_df.empty:
        return pd.DataFrame(
            columns=[
                "run_id", "dataset", "seed", "strategy", "classifier_variant",
                "scorer_variant", "assigner_variant", "metric", "aulc",
                "max_step", "n_points",
            ]
        )

    rows = []
    group_cols = [
        "run_id",
        "dataset",
        "seed",
        "strategy",
        "classifier_variant",
        "scorer_variant",
        "assigner_variant",
        "metric",
    ]
    for keys, group in history_df.groupby(group_cols):
        row = dict(zip(group_cols, keys))
        row["aulc"] = normalized_aulc(group)
        row["max_step"] = float(group["step"].max())
        row["n_points"] = int(group.shape[0])
        rows.append(row)
    return pd.DataFrame(rows)


def summarize_aulc(aulc_df: pd.DataFrame) -> pd.DataFrame:
    if aulc_df.empty:
        return pd.DataFrame(
            columns=["dataset", "strategy", "aulc_mean", "aulc_std", "n_runs", "n_seeds", "seeds"]
        )

    rows = []
    for (dataset, strategy), group in aulc_df.groupby(["dataset", "strategy"], dropna=False):
        n_seeds, seeds = summarize_seed_values(group["seed"])
        rows.append(
            {
                "dataset": dataset,
                "strategy": strategy,
                "aulc_mean": group["aulc"].mean(),
                "aulc_std": group["aulc"].std(),
                "n_runs": int(group["run_id"].nunique()),
                "n_seeds": int(n_seeds),
                "seeds": seeds,
            }
        )
    return pd.DataFrame(rows).sort_values(["dataset", "aulc_mean"], ascending=[True, False])


def summarize_aulc_overall(aulc_df: pd.DataFrame) -> pd.DataFrame:
    if aulc_df.empty:
        return pd.DataFrame(
            columns=["strategy", "aulc_mean", "aulc_std", "n_runs", "n_seeds", "seeds"]
        )

    rows = []
    for strategy, group in aulc_df.groupby(["strategy"], dropna=False):
        n_seeds, seeds = summarize_seed_values(group["seed"])
        rows.append(
            {
                "strategy": strategy,
                "aulc_mean": group["aulc"].mean(),
                "aulc_std": group["aulc"].std(),
                "n_runs": int(group["run_id"].nunique()),
                "n_seeds": int(n_seeds),
                "seeds": seeds,
            }
        )
    return pd.DataFrame(rows).sort_values("aulc_mean", ascending=False)


def plot_learning_curves(
    history_df: pd.DataFrame,
    *,
    strategies=None,
    dataset_order=None,
    title=None,
):
    df = history_df.copy()
    if strategies is not None:
        df = df[df["strategy"].isin(strategies)]
    if df.empty:
        return None

    datasets = dataset_order or list(df["dataset"].dropna().unique())
    if not datasets:
        return None

    agg = (
        df.groupby(["dataset", "strategy", "step"], as_index=False)
        .agg(mean_value=("value", "mean"), std_value=("value", "std"), n=("value", "count"))
    )
    if agg.empty:
        return None

    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 4.5), sharey=True)
    if len(datasets) == 1:
        axes = [axes]

    for ax, dataset in zip(axes, datasets):
        ds_df = agg[agg["dataset"] == dataset]
        for strategy, group in ds_df.groupby("strategy"):
            group = group.sort_values("step")
            ax.plot(group["step"], group["mean_value"], label=strategy)
            err = group["std_value"].fillna(0.0)
            ax.fill_between(group["step"], group["mean_value"] - err, group["mean_value"] + err, alpha=0.15)
        ax.set_title(dataset)
        ax.set_xlabel("Labeled pair budget (MLflow step)")
        ax.set_ylabel(df["metric"].iloc[0])

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5))
    if title:
        fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


def pairwise_win_rates(history_df: pd.DataFrame, *, strategies=None) -> pd.DataFrame:
    output_cols = [
        "strategy_a", "strategy_b", "n_cells", "win_rate_a",
        "tie_rate", "loss_rate_a", "mean_delta", "median_delta",
    ]
    df = history_df.copy()
    if strategies is not None:
        df = df[df["strategy"].isin(strategies)]
    if df.empty:
        return pd.DataFrame(columns=output_cols)

    pivot = df.pivot_table(index=["dataset", "seed", "step"], columns="strategy", values="value")
    available = list(pivot.columns)
    if len(available) < 2:
        return pd.DataFrame(columns=output_cols)

    rows = []
    for a, b in itertools.permutations(available, 2):
        pair = pivot[[a, b]].dropna()
        if pair.empty:
            continue
        delta = pair[a] - pair[b]
        rows.append(
            {
                "strategy_a": a,
                "strategy_b": b,
                "n_cells": int(pair.shape[0]),
                "win_rate_a": float((delta > 0).mean()),
                "tie_rate": float((delta == 0).mean()),
                "loss_rate_a": float((delta < 0).mean()),
                "mean_delta": float(delta.mean()),
                "median_delta": float(delta.median()),
            }
        )
    if not rows:
        return pd.DataFrame(columns=output_cols)
    return pd.DataFrame(rows, columns=output_cols).sort_values(
        ["strategy_a", "win_rate_a", "mean_delta"], ascending=[True, False, False]
    )


def pairwise_win_matrix(pairwise_df: pd.DataFrame, value_col: str = "win_rate_a") -> pd.DataFrame:
    if pairwise_df.empty:
        return pd.DataFrame()
    matrix = pairwise_df.pivot(index="strategy_a", columns="strategy_b", values=value_col)
    for strategy in sorted(set(matrix.index).union(matrix.columns)):
        matrix.loc[strategy, strategy] = np.nan
    return matrix.sort_index().sort_index(axis=1)


In [ ]:
client = configure_tracking(RESULTS_PATH)
experiment = find_experiment(client, EXPERIMENT_NAME)
runs = search_runs_all(client, experiment.experiment_id)
runs_df = enrich_runs(runs_to_frame(runs))
expected_rows = count_manifest_rows(MANIFEST_PATH)
use_case = load_use_case(USE_CASE_PATH)
expected_seed_values = extract_expected_seed_values(use_case)

print(f"Experiment ID         : {experiment.experiment_id}")
print(f"Experiment name       : {experiment.name}")
print(f"Expected manifest rows: {expected_rows}")
print(f"Observed MLflow runs  : {len(runs_df)}")
print(f"Configured seeds      : {expected_seed_values}")
print(f"Configured seed count : {len(expected_seed_values) if expected_seed_values is not None else '?'}")

status_overview = runs_df.groupby(["dataset", "status"]).size().rename("n_runs").reset_index()
display(status_overview.sort_values(["dataset", "status"]))

coverage_overview = summarize_run_coverage(runs_df)
display(coverage_overview.head(30))

strategy_overview = (
    runs_df.groupby(["dataset", "strategy", "status"], as_index=False)
    .agg(
        n_runs=("run_id", "nunique"),
        n_seeds=("seed", lambda s: summarize_seed_values(s)[0]),
    )
    .sort_values(["dataset", "strategy", "status"])
)
display(strategy_overview.head(30))


In [ ]:
# Optional exact-match filters. Use [] or None to disable a filter.
RUN_FILTERS = {
    "dataset": [],
    "seed": [],
    "classifier_variant": ["annot_mix[e=8/8]", "crowd_em", "majority_voting"],
    "scorer_variant": [],
    "assigner_variant": ["quota_sample", "epsilon_greedy_sample[eps=0.1]", "epsilon_greedy_sample[eps=0.05]", "greedy_sample"],
    "strategy": [],
}

# Optional substring filter over the full strategy label. All tokens must match.
STRATEGY_CONTAINS = []

# Restrict the evaluation subset shown below.
FILTER_ONLY_FINISHED = ONLY_FINISHED

print("Available datasets:", sorted(runs_df["dataset"].dropna().unique().tolist()))
print("Available classifier variants:")
for value in sorted(runs_df["classifier_variant"].dropna().unique().tolist()):
    print(" -", value)
print("Available scorer variants:", sorted(runs_df["scorer_variant"].dropna().unique().tolist()))
print("Available assigner variants:", sorted(runs_df["assigner_variant"].dropna().unique().tolist()))

filtered_runs_raw_df = apply_run_filters(
    runs_df,
    filters=RUN_FILTERS,
    strategy_contains=STRATEGY_CONTAINS,
    only_finished=FILTER_ONLY_FINISHED,
)

print(f"Filtered raw runs: {len(filtered_runs_raw_df)} / {len(runs_df)}")
display(summarize_run_coverage(filtered_runs_raw_df).head(30))

if DEDUPLICATE_LATEST_PER_SEED:
    filtered_runs_df = select_latest_seed_runs(filtered_runs_raw_df)
    print(
        f"Deduplicated to latest run per dataset/strategy/seed: "
        f"{len(filtered_runs_df)} / {len(filtered_runs_raw_df)}"
    )
else:
    filtered_runs_df = filtered_runs_raw_df.copy()

filtered_coverage = summarize_run_coverage(filtered_runs_df)
display(filtered_coverage.head(30))

display(
    filtered_runs_df[["run_id", "dataset", "seed", "status", "classifier_variant", "scorer_variant", "assigner_variant", "strategy"]]
    .head(20)
)


In [ ]:
history_df = load_metric_history_table(client, filtered_runs_df, PRIMARY_METRIC, only_finished=FILTER_ONLY_FINISHED)
print(
    f"Loaded {len(history_df)} metric-history rows for metric={PRIMARY_METRIC!r} "
    f"from {len(filtered_runs_df)} filtered runs."
)

history_run_coverage = summarize_run_coverage(
    history_df[["run_id", "dataset", "seed", "strategy"]].drop_duplicates()
)
print("History coverage after metric loading:")
display(history_run_coverage)

aulc_df = compute_aulc_per_run(history_df)
aulc_by_dataset = summarize_aulc(aulc_df)
aulc_overall = summarize_aulc_overall(aulc_df)

display(aulc_overall)
display(aulc_by_dataset)

plot_strategies = sorted(history_df["strategy"].dropna().unique().tolist()) if not history_df.empty else []
print(f"Plotting {len(plot_strategies)} filtered strategies.")
for strategy in plot_strategies:
    print(" -", strategy)

dataset_order = [d for d in ["letter26", "trec6", "dopanim"] if d in history_df["dataset"].unique()] if not history_df.empty else []

fig = plot_learning_curves(
    history_df,
    strategies=plot_strategies,
    dataset_order=dataset_order,
    title=f"{PRIMARY_METRIC} learning curves (all filtered strategies)",
)
if fig is not None:
    plt.show()
else:
    print("No learning curves available for the current filtered subset.")

if not aulc_by_dataset.empty:
    aulc_pivot = aulc_by_dataset.pivot(index="strategy", columns="dataset", values="aulc_mean")
    display(aulc_pivot.sort_index())
else:
    print("No AULC rows available.")


In [ ]:
pairwise_df = pairwise_win_rates(history_df, strategies=plot_strategies)
display(pairwise_df)

win_matrix = pairwise_win_matrix(pairwise_df)
if not win_matrix.empty:
    display(win_matrix.reindex(index=plot_strategies, columns=plot_strategies))
else:
    print("No pairwise comparisons available for the current filtered strategies.")

reference_strategy = plot_strategies[0] if plot_strategies else None
if reference_strategy is not None and not pairwise_df.empty:
    vs_reference = pairwise_df[pairwise_df["strategy_a"] == reference_strategy].copy()
    display(vs_reference.sort_values(["win_rate_a", "mean_delta"], ascending=False))
